# 03 - Pump anomaly detection on platform metrics

Notebook 01 found a RandomForest that detects SKAB pump faults from all eight
lab sensors (held-out F1 0.733, recall 0.60). The platform's devices don't
report those eight sensors. They report the metric catalogue: vibration
acceleration, pressure, temperature, flow rate and power. This notebook
retrains the same model on **only what a platform device sends**, so that the
Spring API can compute every feature from a batch of readings and
`apps/ml` can score it (`.spec/features/failure-risk.md`, phase 0b).

01 stays as the research record. This notebook reuses its split,
cross-validation and held-out protocol unchanged.

Outline: mapping -> loading -> features (Spring-computable) -> split ->
CV and threshold -> held-out test and comparison with 01 -> Faulty threshold
-> ONNX export, parity fixture and simulation dataset.

## 1. Mapping SKAB to platform metrics

Units are from SKAB's README (<https://github.com/waico/SKAB>):

| SKAB column | Unit | Platform metric (`AnomalyCheckRequest` field) |
| --- | --- | --- |
| `Accelerometer1RMS` | g (RMS) | `vibrationAcceleration` (g RMS) |
| `Pressure` | bar | `pressure` (bar) |
| `Temperature` | °C (pump body) | `temperature` (°C) |
| `Volume Flow RateRMS` | L/min (circulation flow rate) | `flowRate` (L/min) |
| `Current` × `Voltage` / 1000 | A × V | `power` (kW) |

- **Flow rate unit confirmed:** SKAB documents `Volume Flow RateRMS` as the
  circulation flow rate in litres per minute, so it maps one to one.
  Its values sit around 32, which fits a small lab loop in L/min.
- **Power ignores power factor.** `Current × Voltage` is *apparent* power
  (kVA after dividing by 1000), not real power. SKAB gives no power factor, so
  the model learns on apparent power. A real meter reporting real power (kW)
  reads lower by the motor's power factor (often 0.7-0.9). Treat this as a
  known domain gap.
- **Dropped:** `Accelerometer2RMS` (a second accelerometer; a platform device
  has one) and `Thermocouple` (fluid temperature; the catalogue has one
  temperature).
- **Licence:** SKAB is GPL-3.0. The model trained on it is used for
  education only.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "skab"
ARTIFACT_DIR = ROOT / "artifacts" / "pump-anomaly-platform"
HANDOFF_PATH = ROOT.parent / "apps" / "ml" / "models" / "model.onnx"

SKAB_SENSORS = [
    "Accelerometer1RMS",
    "Accelerometer2RMS",
    "Current",
    "Pressure",
    "Temperature",
    "Thermocouple",
    "Voltage",
    "Volume Flow RateRMS",
]
# Platform metrics, named as AnomalyCheckRequest's reading fields. This is
# also the order of the raw block in the feature vector.
METRICS = ["vibrationAcceleration", "pressure", "temperature", "flowRate", "power"]

# Same files and seed as notebook 01.
FILES = (
    ["anomaly-free/anomaly-free.csv"]
    + [f"valve1/{i}.csv" for i in range(16)]
    + [f"valve2/{i}.csv" for i in range(4)]
    + [f"other/{i}.csv" for i in range(1, 15)]
)
SEED = 42

## 2. Loading

Reads the CSVs notebook 01 downloaded into `data/skab/` (run 01 first on a
fresh checkout), then adds the five platform columns.

In [ ]:
def load(files: list[str]) -> pd.DataFrame:
    frames = []
    for rel in files:
        df = pd.read_csv(DATA_DIR / rel, sep=";", index_col="datetime", parse_dates=True)
        for label in ("anomaly", "changepoint"):
            if label not in df:
                df[label] = 0.0
        df["experiment"] = rel
        df["group"] = rel.split("/")[0]
        frames.append(df)
    return pd.concat(frames)


def to_platform(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "vibrationAcceleration": df["Accelerometer1RMS"],
            "pressure": df["Pressure"],
            "temperature": df["Temperature"],
            "flowRate": df["Volume Flow RateRMS"],
            # Apparent power: ignores power factor (section 1).
            "power": df["Current"] * df["Voltage"] / 1000,
        },
        index=df.index,
    )
    return pd.concat([df, out], axis=1)


missing = [f for f in FILES if not (DATA_DIR / f).exists()]
assert not missing, f"Run notebook 01 first to download SKAB: {missing[:3]}"
raw = to_platform(load(FILES))
raw["anomaly"] = raw["anomaly"].astype(int)
raw[METRICS].describe().T

## 3. Features Spring can compute

Every feature must be computable by the Spring API from the readings in one
batch, with no stored state (`.spec/features/failure-risk.md`, Anomaly
detection). So the feature set is notebook 01's **base** set, restricted to
the five metrics: per metric, the raw value plus the rolling mean and
standard deviation over the last `WINDOW` = 10 readings. 15 features in all.

**Exact definition, for the Spring implementation.** Readings are sorted by
timestamp and indexed `i = 0, 1, ...` within the batch. For each metric `x`:

- `window(i)` is readings `max(0, i - 9)` to `i` inclusive, so
  `n = min(i + 1, 10)` readings. Row count, not time.
- `x_mean(i) = sum(window) / n`.
- `x_std(i)` is the **sample** standard deviation (ddof = 1, pandas' default):
  `sqrt(sum((x_j - x_mean(i))^2) / (n - 1))`. For `i = 0` (`n = 1`) it is
  undefined, and the feature is **0**.
- **The first 9 readings of a batch use partial windows** (`n` = 1 to 9);
  nothing is dropped or padded. Windows **restart at every batch**, exactly
  as they restart at every SKAB file here.
- Compute in double precision. `apps/ml` casts the inputs to float32.

Notebook 01 dropped each file's first 90 rows, because its extended features
needed a baseline. These features don't, and a real batch scores from its
first reading, so **every row is kept** here: the model sees partial windows
in training, as it will in production. For the comparison with 01 (section
6), both models are also scored on 01's rows only.

In [ ]:
WINDOW = 10  # readings, about 10 s at ~1 Hz
BASELINE_ROWS = 90  # notebook 01's dropped opening rows, for comparison only


def add_features(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Raw + rolling mean/std (ddof=1, min_periods=1, first std = 0) per file."""
    out = df.reset_index()
    by_file = out.groupby("experiment", sort=False)[cols]
    mean = by_file.transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())
    std = by_file.transform(lambda s: s.rolling(WINDOW, min_periods=1).std(ddof=1)).fillna(0)
    out = pd.concat([out, mean.add_suffix("_mean"), std.add_suffix("_std")], axis=1)
    out["row"] = out.groupby("experiment", sort=False).cumcount()
    return out


def feature_names(cols: list[str]) -> list[str]:
    return cols + [f"{c}_mean" for c in cols] + [f"{c}_std" for c in cols]


PLATFORM = feature_names(METRICS)  # the shipped model's inputs
SKAB8 = feature_names(SKAB_SENSORS)  # notebook 01's base set, for comparison
FEATURE_SETS = {"platform": PLATFORM, "skab8": SKAB8}

data = add_features(raw, METRICS + SKAB_SENSORS)
assert data[PLATFORM + SKAB8].notna().all().all()
print(len(PLATFORM), "platform features:", PLATFORM)

**Sampling gaps.** The endpoint rejects a batch with a gap over 2 seconds.
SKAB is mostly 1-2 s apart, but a few files have longer gaps. Training keeps
one window per file, as 01 did; the gaps are listed so the simulation dataset
(section 9) can be split at them.

In [ ]:
gap = data.groupby("experiment", sort=False)["datetime"].diff().dt.total_seconds()
data["gap_s"] = gap
long_gaps = data[gap > 2].groupby("experiment")["gap_s"].agg(["count", "max"])
long_gaps

## 4. Train / test split by experiment

Identical to notebook 01: with `SEED = 42`, a quarter of each group's
labelled files is held out whole, and `anomaly-free.csv` always trains.
The assert pins the split to 01's recorded test files.

In [ ]:
rng = np.random.default_rng(SEED)
labelled = [f for f in FILES if not f.startswith("anomaly-free")]
test_files = []
for group in ("valve1", "valve2", "other"):
    members = [f for f in labelled if f.startswith(group)]
    n_test = max(1, len(members) // 4)
    test_files += [str(f) for f in rng.choice(members, size=n_test, replace=False)]

nb01 = json.loads((ROOT / "artifacts" / "skab-pump-anomaly" / "metrics.json").read_text())
assert sorted(test_files) == nb01["test_files"], "split differs from notebook 01"

is_test = data["experiment"].isin(test_files)
train, test = data[~is_test], data[is_test]
y_train, y_test = train["anomaly"], test["anomaly"]
print("test files:", sorted(test_files))
print("train rows:", len(train), "anomaly rate:", round(y_train.mean(), 3))
print("test rows: ", len(test), "anomaly rate:", round(y_test.mean(), 3))

## 5. Model, cross-validation and decision threshold

The model is notebook 01's winner, unchanged: a `RandomForestClassifier`
(200 trees, depth 12, balanced class weights, no scaler so the ONNX export is
exact). Notebook 01 already compared model families; this notebook only
changes the inputs. The 8-sensor model is retrained alongside it, on the same
rows and folds, so the two are compared like for like.

As in 01: 5-fold `GroupKFold` over the labelled training files, with
`anomaly-free.csv` on the fitting side of every fold, scored at 0.5. The
5-input model's decision threshold is the one maximising F1 on its
out-of-fold scores. The held-out files play no part.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline


def make_rf() -> Pipeline:
    return Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=12,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=SEED,
                ),
            ),
        ]
    )


def evaluate(y_true, y_pred, scores) -> dict[str, float]:
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, scores),
    }


CANDIDATES = {"RF / 5 platform metrics": "platform", "RF / 8 SKAB sensors": "skab8"}
SHIPPED = "RF / 5 platform metrics"

cv_files = [f for f in labelled if f not in test_files]
cv_data = train[train["experiment"].isin(cv_files)].reset_index(drop=True)
free = train[~train["experiment"].isin(cv_files)]

fold_rows = []
oof = pd.DataFrame({"experiment": cv_data["experiment"], "anomaly": cv_data["anomaly"]})
for fold, (fit_idx, eval_idx) in enumerate(
    GroupKFold(n_splits=5).split(cv_data, groups=cv_data["experiment"])
):
    fit = pd.concat([free, cv_data.iloc[fit_idx]])
    held = cv_data.iloc[eval_idx]
    for name, feature_set in CANDIDATES.items():
        cols = FEATURE_SETS[feature_set]
        scores = make_rf().fit(fit[cols], fit["anomaly"]).predict_proba(held[cols])[:, 1]
        pred = (scores >= 0.5).astype(int)
        oof.loc[held.index, f"{name} score"] = scores
        fold_rows.append({"fold": fold, "model": name} | evaluate(held["anomaly"], pred, scores))

cv_results = pd.DataFrame(fold_rows)
cv_summary = cv_results.groupby("model", sort=False)[
    ["precision", "recall", "f1", "roc_auc"]
].agg(["mean", "std"])
cv_summary.round(3)

In [ ]:
precision, recall, thresholds = precision_recall_curve(oof["anomaly"], oof[f"{SHIPPED} score"])
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
best = np.argmax(f1[:-1])
THRESHOLD = round(float(thresholds[best]), 2)
print(f"decision threshold: {THRESHOLD}  (OOF F1 {f1[best]:.3f})")

## 6. Held-out test, and what was lost against notebook 01

Both models are refitted on all training files and score the held-out files
once. The 5-input model uses its tuned threshold; the 8-sensor model uses
01's recorded threshold (0.49).

Two row sets:
- **all rows**: every reading, partial windows included (what production sees);
- **01's rows**: each file's first 90 rows removed, the rows notebook 01's
  F1 0.733 was measured on.

In [ ]:
NB01_THRESHOLD = nb01["decision_threshold"]
thresholds_by_model = {SHIPPED: THRESHOLD, "RF / 8 SKAB sensors": NB01_THRESHOLD}
nb01_rows = test["row"] >= BASELINE_ROWS

results, fitted, test_scores = {}, {}, {}
for name, feature_set in CANDIDATES.items():
    cols = FEATURE_SETS[feature_set]
    fitted[name] = make_rf().fit(train[cols], y_train)
    scores = fitted[name].predict_proba(test[cols])[:, 1]
    test_scores[name] = scores
    pred = (scores >= thresholds_by_model[name]).astype(int)
    for rows_name, mask in (("all rows", np.ones(len(test), bool)), ("01's rows", nb01_rows.to_numpy())):
        results[(name, rows_name)] = evaluate(y_test[mask], pred[mask], scores[mask])

model = fitted[SHIPPED]
win_scores = test_scores[SHIPPED]
win_pred = (win_scores >= THRESHOLD).astype(int)
comparison = pd.DataFrame(results).T.round(3)
comparison

In [ ]:
per_file = (
    pd.DataFrame(
        {
            "experiment": test["experiment"].to_numpy(),
            "anomaly": y_test.to_numpy(),
            SHIPPED: win_pred,
            "RF / 8 SKAB sensors": (test_scores["RF / 8 SKAB sensors"] >= NB01_THRESHOLD).astype(int),
        }
    )
    .query("anomaly == 1")
    .groupby("experiment")[[SHIPPED, "RF / 8 SKAB sensors"]]
    .mean()
)
print("Per-file recall on held-out experiments")
per_file.round(2)

In [ ]:
importances = pd.Series(model.named_steps["model"].feature_importances_, index=PLATFORM)
importances.sort_values().plot.barh(figsize=(7, 5), title="5-input RF feature importance")
plt.tight_layout()
plt.show()

**Reading the comparison.** Numbers from this run (re-read the tables on a
re-run; `README.md` is generated from them):

- **Against notebook 01's recorded result** (8 sensors, held-out F1 0.733,
  precision 0.94, recall 0.60): the 5-input model scores F1 0.744 on the same
  rows (precision 0.88, recall 0.65) and 0.727 on all rows. **Little or no F1
  was lost.** What was lost is precision: about 0.94 -> 0.88 on 01's rows, so
  more false alarms per reading.
- The 8-sensor model retrained here scores 0.706, not 0.733, on 01's rows.
  The difference is that this notebook also trains on each file's first 90
  rows (with partial windows). So the 8-vs-5 comparison inside this notebook
  is like for like, and 01's 0.733 is the reference for the shipped 8-sensor
  research model.
- In CV the two are tied (F1 0.775 ± 0.101 vs 0.789 ± 0.090), and the test
  set is 8 files. The 5-input model doing slightly better on the test files
  (mostly `valve1/1`) is noise, not a sign that fewer inputs help.
- The dropped inputs (second accelerometer, fluid thermocouple, separate
  current and voltage) carried little that the five metrics don't. The
  same files are missed by both: `other/8` (recall 0.01) and `other/3` (0.32).
- **Still useful?** Yes, for the platform's purpose. The spec accepts a weaker
  5-input model, and this one is not clearly weaker. The anomaly check only
  records a condition after 10 consecutive abnormal readings, which damps the
  lower per-reading precision. It is not evidence the model works on real
  condo pumps: different hardware, real rather than apparent power, and no
  labels.

## 7. Faulty threshold

The server records a condition when at least 10 consecutive readings are
abnormal (score ≥ decision threshold), and calls it `FAULTY` when **the
median score of the longest abnormal run** reaches the Faulty threshold,
`NEEDS_ATTENTION` otherwise.

**How it is chosen.** `FAULTY` should mean "confidently a real fault". On
the out-of-fold scores (training files only), take the lowest score at which
per-reading **precision reaches 0.95**, rounded up to 2 decimals, and never
below the decision threshold. A run whose median clears it is, reading by
reading, at least 95% likely to be a true anomaly in CV. Below it the run is
real enough to check but not certain: `NEEDS_ATTENTION`. The 0.95 target is
a judgement call to revisit with whoever owns the maintenance workflow.

The cell then applies the server's run rule to the OOF scores, per file, to
show how runs split between the two conditions.

In [ ]:
FAULTY_PRECISION = 0.95
p_oof, r_oof, t_oof = precision_recall_curve(oof["anomaly"], oof[f"{SHIPPED} score"])
reaches = np.flatnonzero(p_oof[:-1] >= FAULTY_PRECISION)
faulty_raw = float(t_oof[reaches[0]]) if len(reaches) else 1.0
FAULTY_AT = max(THRESHOLD, float(np.ceil(faulty_raw * 100) / 100))
print(f"precision >= {FAULTY_PRECISION} from score {faulty_raw:.3f} -> Faulty threshold {FAULTY_AT}")

MIN_RUN = 10


def longest_run(scores: np.ndarray, threshold: float) -> np.ndarray:
    """Scores of the longest run of consecutive readings >= threshold."""
    best, start = (0, 0), None
    for i, abnormal in enumerate(np.append(scores >= threshold, False)):
        if abnormal and start is None:
            start = i
        elif not abnormal and start is not None:
            if i - start > best[1] - best[0]:
                best = (start, i)
            start = None
    return scores[best[0] : best[1]]


def condition_for(scores: np.ndarray) -> str | None:
    run = longest_run(scores, THRESHOLD)
    if len(run) < MIN_RUN:
        return None
    return "FAULTY" if np.median(run) >= FAULTY_AT else "NEEDS_ATTENTION"


run_rows = []
for experiment, part in oof.groupby("experiment"):
    s = part[f"{SHIPPED} score"].to_numpy()
    run = longest_run(s, THRESHOLD)
    run_rows.append(
        {
            "experiment": experiment,
            "anomaly_rate": part["anomaly"].mean(),
            "longest_run": len(run),
            "run_median": np.median(run) if len(run) else np.nan,
            "condition": condition_for(s),
        }
    )
pd.DataFrame(run_rows).round(3)

## 8. Export to ONNX

Same export as notebook 01: one float32 input `input` of shape
`[None, 15]`, `zipmap=False` so `probabilities` is a plain `[n, 2]` tensor
(column 1 = anomaly). Callers compare `probabilities[:, 1]` with the decision
threshold; the ONNX `label` output uses 0.5 and should be ignored.

In [ ]:
import onnxruntime as ort
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

onx = to_onnx(
    model,
    initial_types=[("input", FloatTensorType([None, len(PLATFORM)]))],
    options={id(model.named_steps["model"]): {"zipmap": False}},
    target_opset={"": 17, "ai.onnx.ml": 3},
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "model.onnx"
model_path.write_bytes(onx.SerializeToString())
print(f"model.onnx: {model_path.stat().st_size / 1e6:.2f} MB")

session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
print([(i.name, i.type, i.shape) for i in session.get_inputs()])
print([(o.name, o.type, o.shape) for o in session.get_outputs()])

**sklearn parity.** The forest has no scaler, so ONNX on float32 should
match sklearn to rounding, checked on the whole held-out set.

In [ ]:
X_check = test[PLATFORM].astype(np.float32)
onnx_labels, onnx_proba = session.run(None, {"input": X_check.to_numpy()})
sk_proba = model.predict_proba(X_check)
max_diff = float(np.abs(onnx_proba - sk_proba).max())
decision_agreement = float(((onnx_proba[:, 1] >= THRESHOLD) == (sk_proba[:, 1] >= THRESHOLD)).mean())
print(f"rows: {len(X_check)}, max |proba diff|: {max_diff:.2e}, decision agreement: {decision_agreement:.4f}")
assert max_diff < 1e-3, "ONNX output diverges from sklearn"

**Parity fixture for Spring.** 50 consecutive held-out readings, treated as
**one batch** (windows start fresh at its first reading, as on the server),
straddling the start of an anomaly so both normal and abnormal scores
appear. It records the input readings, the expected 15 features per reading
(float64, in feature order) and the expected ONNX anomaly scores.
`AnomalyFeatureTests` should match the features to about 1e-9 and the scores
to about 1e-5.

In [ ]:
FIXTURE_ROWS = 50
fixture_file = "valve1/1.csv"
part = test[test["experiment"] == fixture_file].reset_index(drop=True)
onset = int(part["anomaly"].to_numpy().argmax())
batch = part.iloc[onset - FIXTURE_ROWS // 2 : onset + FIXTURE_ROWS // 2]
assert batch["gap_s"].iloc[1:].max() <= 2, "fixture batch must have no gap over 2 s"

batch_raw = batch.set_index("datetime")[METRICS + ["anomaly"]].assign(experiment="fixture")
batch_features = add_features(batch_raw, METRICS)[PLATFORM]
fixture_scores = session.run(None, {"input": batch_features.to_numpy(np.float32)})[1][:, 1]


def iso(ts: pd.Timestamp) -> str:
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


fixture = {
    "source": f"{fixture_file}, rows {onset - FIXTURE_ROWS // 2}-{onset + FIXTURE_ROWS // 2 - 1}, scored as one batch",
    "window_rows": WINDOW,
    "std": "sample standard deviation (ddof=1) over the window; 0 when the window has one reading",
    "feature_order": PLATFORM,
    "decision_threshold": THRESHOLD,
    "rows": [
        {
            "reading": {"timestamp": iso(ts)} | {m: float(r[m]) for m in METRICS},
            "anomaly": int(r["anomaly"]),
            "features": [float(v) for v in f],
            "score": float(s),
        }
        for (ts, r), f, s in zip(
            batch_raw.iterrows(), batch_features.to_numpy(), fixture_scores, strict=True
        )
    ],
}
(ARTIFACT_DIR / "parity-fixture.json").write_text(json.dumps(fixture, indent=2))
print("fixture abnormal readings:", int((fixture_scores >= THRESHOLD).sum()), "of", FIXTURE_ROWS)
print("condition the server would record:", condition_for(fixture_scores))

## 9. Simulation dataset

The held-out experiments as raw readings in platform units, for driving
simulated pumps against `POST .../readings/anomaly-check`. Reading columns
use the `AnomalyCheckRequest` field names; `anomaly` is SKAB's ground truth.
Ordered by experiment, then timestamp.

SKAB timestamps have no time zone; they are written as UTC (`...Z`). The
files with gaps over 2 seconds (section 3) must be split into separate
batches at those gaps, or the endpoint rejects the batch with 400.

In [ ]:
sim = test.sort_values(["experiment", "datetime"])[["experiment", "datetime"] + METRICS + ["anomaly"]]
sim = sim.rename(columns={"datetime": "timestamp"})
sim["timestamp"] = sim["timestamp"].map(iso)
assert list(sim.columns) == [
    "experiment", "timestamp", "vibrationAcceleration", "pressure",
    "temperature", "flowRate", "power", "anomaly",
]
sim.to_csv(ARTIFACT_DIR / "test-readings.csv", index=False)

sim_summary = test.groupby("experiment").agg(
    readings=("anomaly", "size"),
    anomaly_rate=("anomaly", "mean"),
    gaps_over_2s=("gap_s", lambda s: int((s > 2).sum())),
)
sim_summary.round(3)

## 10. Metrics, README and hand-off

`metrics.json` carries what the Spring constants are copied from: the
feature order, the decision threshold and the Faulty threshold. The model is
then copied to `apps/ml/models/model.onnx`, which `apps/ml` serves.

In [ ]:
import shutil

def rounded(d: dict) -> dict:
    return {m: round(float(v), 4) for m, v in d.items()}

shipped_cv = cv_summary.loc[SHIPPED]
test_all = results[(SHIPPED, "all rows")]
test_01 = results[(SHIPPED, "01's rows")]
skab8_01 = results[("RF / 8 SKAB sensors", "01's rows")]
nb01_rec = next(v for k, v in nb01["metrics_test"].items() if "WINNER" in k)
estimator = model.named_steps["model"]

metrics = {
    "dataset": "SKAB (https://github.com/waico/SKAB), GPL-3.0, education use only",
    "notebook": "notebooks/03-pump-anomaly-platform-metrics.ipynb",
    "model": type(estimator).__name__,
    "params": estimator.get_params(),
    "inputs": {
        "vibrationAcceleration": "Accelerometer1RMS (g RMS)",
        "pressure": "Pressure (bar)",
        "temperature": "Temperature (degC)",
        "flowRate": "Volume Flow RateRMS (L/min)",
        "power": "Current * Voltage / 1000 (kW; apparent power, ignores power factor)",
    },
    "n_features": len(PLATFORM),
    "feature_order": PLATFORM,
    "features": {
        "window_rows": WINDOW,
        "mean": "mean of readings max(0, i-9)..i within the batch",
        "std": "sample std (ddof=1) of the same window; 0 when the window has one reading",
        "batch_start": "windows restart at each batch; the first 9 readings use partial windows",
    },
    "seed": SEED,
    "test_files": sorted(test_files),
    "onnx_input": {"name": "input", "type": "float32", "shape": [None, len(PLATFORM)]},
    "onnx_outputs": ["label (int64, ignore: uses 0.5)", "probabilities (float32, [n, 2]; column 1 = anomaly)"],
    "decision_threshold": THRESHOLD,
    "decision_threshold_source": "best F1 on 5-fold GroupKFold out-of-fold predictions (training files only)",
    "faulty_threshold": FAULTY_AT,
    "faulty_threshold_source": (
        f"lowest out-of-fold score with per-reading precision >= {FAULTY_PRECISION}, "
        "rounded up to 2 dp, not below the decision threshold; compared with the median "
        "score of the longest abnormal run"
    ),
    "min_abnormal_run": MIN_RUN,
    "onnx_parity": {"rows": len(X_check), "max_abs_proba_diff": max_diff, "decision_agreement": decision_agreement},
    "metrics_cv_at_0_5": {
        name: rounded(row) for name, row in cv_results.groupby("model")[["precision", "recall", "f1", "roc_auc"]].mean().iterrows()
    },
    "metrics_test": {f"{n} @ {thresholds_by_model[n]} / {rows}": rounded(r) for (n, rows), r in results.items()},
}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))

size_mb = model_path.stat().st_size / 1e6
experiment_lines = "\n".join(
    f"  - `{e}`: {int(r.readings)} readings, anomaly rate {r.anomaly_rate:.2f}"
    + (f", {int(r.gaps_over_2s)} gap(s) over 2 s" if r.gaps_over_2s else "")
    for e, r in sim_summary.iterrows()
)
readme = f"""# pump-anomaly-platform

Produced by `notebooks/03-pump-anomaly-platform-metrics.ipynb` (phase 0b of
`.spec/features/failure-risk.md`). Served by `apps/ml` as
`apps/ml/models/model.onnx`.

## Licence

SKAB (https://github.com/waico/SKAB) is **GPL-3.0**. This model is trained on
it and is used for **education only**.

## Model

- RandomForestClassifier (200 trees, depth 12, balanced), seed {SEED}; {size_mb:.1f} MB.
- Inputs, mapped from SKAB: vibrationAcceleration = Accelerometer1RMS (g RMS),
  pressure (bar), temperature (degC), flowRate = Volume Flow RateRMS (L/min),
  power = Current x Voltage / 1000 (kW; apparent power, **ignores power factor**).
  Accelerometer2RMS and Thermocouple are dropped.
- Features ({len(PLATFORM)}, order in `metrics.json` -> `feature_order`): per metric
  the raw value, then the rolling mean, then the rolling std over the last
  {WINDOW} readings of the batch. Std is the sample std (ddof = 1), and 0 for
  a single reading. Windows restart at each batch; its first 9 readings use
  partial windows.
- ONNX input `input`, float32 `[None, {len(PLATFORM)}]`. Use `probabilities[:, 1]`;
  ignore `label` (it uses 0.5). sklearn parity: max |diff| {max_diff:.1e} over
  {len(X_check)} held-out rows.

## Thresholds

- Decision threshold **{THRESHOLD}**: a reading is abnormal at or above it (best
  F1 on out-of-fold CV scores).
- Faulty threshold **{FAULTY_AT}**: a run of >= {MIN_RUN} abnormal readings is
  FAULTY when its median score reaches it, NEEDS_ATTENTION otherwise. Chosen
  as the lowest out-of-fold score with per-reading precision >= {FAULTY_PRECISION}.

## Results (held-out experiments)

| Model | Rows | Precision | Recall | F1 | ROC-AUC |
| --- | --- | --- | --- | --- | --- |
| 5 platform metrics @ {THRESHOLD} | all | {test_all['precision']:.3f} | {test_all['recall']:.3f} | {test_all['f1']:.3f} | {test_all['roc_auc']:.3f} |
| 5 platform metrics @ {THRESHOLD} | 01's (row >= 90) | {test_01['precision']:.3f} | {test_01['recall']:.3f} | {test_01['f1']:.3f} | {test_01['roc_auc']:.3f} |
| 8 SKAB sensors @ {NB01_THRESHOLD}, retrained here | 01's (row >= 90) | {skab8_01['precision']:.3f} | {skab8_01['recall']:.3f} | {skab8_01['f1']:.3f} | {skab8_01['roc_auc']:.3f} |
| 8 SKAB sensors, notebook 01 as recorded | 01's (row >= 90) | {nb01_rec['precision']:.3f} | {nb01_rec['recall']:.3f} | {nb01_rec['f1']:.3f} | {nb01_rec['roc_auc']:.3f} |

CV (5-fold over training files, at 0.5): F1 {shipped_cv[('f1', 'mean')]:.3f} ± {shipped_cv[('f1', 'std')]:.3f}.

## Files

- `model.onnx` - the model.
- `metrics.json` - feature order, thresholds, feature definitions, metrics.
- `parity-fixture.json` - {FIXTURE_ROWS} readings scored as one batch, with their
  expected features and scores, for the Spring `AnomalyFeatureTests`.
- `test-readings.csv` - the held-out experiments as raw readings
  (`experiment,timestamp,vibrationAcceleration,pressure,temperature,flowRate,power,anomaly`),
  ordered by experiment then timestamp. Timestamps are SKAB's local times
  written as UTC. Split a file into separate batches at any gap over 2 s.
  Experiments:
{experiment_lines}
"""
(ARTIFACT_DIR / "README.md").write_text(readme)

HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(model_path, HANDOFF_PATH)
print(readme)